<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/01_local_llm_colab_gpu.ipynb)


# colab — Local LLM on Free GPU

## _Running a Small Open-Weight Model with Google Colab’s T4 / L4 GPU_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Download and run a small, open-weight LLM locally on a free Colab GPU.
- **Hardware**: Designed for Google Colab (T4 default, L4 optional). GPU is
    auto-detected.
- **Model**: We use `Qwen2.5-1.5B-Instruct` (1.5 billion parameters) which fits
    comfortably into the ~16 GB VRAM of a T4.
- **Why local?**: No API keys, no data leaves the notebook, full control over the
    inference loop.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the full GPU comparison table.

### Roadmap

1. **Environment**: Check the GPU and install `transformers`, `torch`, and
     `accelerate`.
2. **Download**: Load the model weights from Hugging Face directly into GPU memory.
3. **Chat**: Run a simple interactive chat loop with system prompts and user input.
4. **Extras**: Show batched generation and a quick throughput benchmark.

### Environment Check

Verify that a GPU is available and install the minimal set of Python packages needed
for local inference.


In [ ]:
#@title Check GPU and install dependencies
!nvidia-smi
!pip -q install transformers accelerate torch huggingface_hub

In [ ]:
#@title Detect GPU and print memory summary
import subprocess
import torch

_out = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
).strip()

print("GPU:", _out)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Current device:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

### Download and Load the Model

We load `Qwen/Qwen2.5-1.5B-Instruct` directly onto the GPU in `bfloat16` (fallback to
`float16` on older GPUs).

**Model specs**
- Parameters: 1.5 B
- Weights: ~3 GB in bf16
- Fits easily into a Colab T4 (16 GB VRAM) with room for a long context window.

In [ ]:
#@title Load model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True,
).to(device) if device == "cpu" else AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded on {device} with dtype {dtype}")
if device == "cuda":
    vram = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM allocated: {vram:.2f} GB")
else:
    print("CPU mode")

### Interactive Chat

We build a small chat helper that formats messages the way the model expects them
(chat templates) and streams the response.


In [ ]:
#@title Chat helper
def chat(messages, max_new_tokens=256, temperature=0.7, top_p=0.9):
    """Run a chat turn and return the assistant's reply."""
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
    )
    return response.strip()

# Seed conversation with a system prompt
conversation = [
    {
        "role": "system",
        "content": "You are a helpful coding assistant. "
        "Keep answers short and precise.",
    },
]

In [ ]:
#@title Demo conversation
user_question = "Write a one-liner in Python to reverse a string."

conversation.append({"role": "user", "content": user_question})
answer = chat(conversation)
conversation.append({"role": "assistant", "content": answer})

print("User:", user_question)
print("Assistant:", answer)

### Extras: Batch Generation and Benchmark

1. **Batch generation** — process multiple prompts in one forward pass for higher
     throughput.
2. **Quick benchmark** — measure tokens-per-second on a fixed prompt to verify GPU
     utilisation.


In [ ]:
#@title Batch generation example
prompts = [
    "Explain recursion in one sentence.",
    "Name three Python built-in data structures.",
    "What is a GPU good for in machine learning?",
]

batched_inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
).to(model.device)

batched_outputs = model.generate(
    **batched_inputs,
    max_new_tokens=64,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
)

for i, out in enumerate(batched_outputs):
    start = batched_inputs.attention_mask[i].sum().item()
    text = tokenizer.decode(out[start:], skip_special_tokens=True)
    print(f"--- Prompt {i+1} ---")
    print(text.strip())
    print()

In [ ]:
#@title Quick throughput benchmark
import time

bench_prompt = "The future of artificial intelligence is"
bench_input = tokenizer(bench_prompt, return_tensors="pt").to(model.device)

# Warm-up
_ = model.generate(**bench_input, max_new_tokens=20, do_sample=False)
torch.cuda.synchronize() if device == "cuda" else None

# Timed run
start = time.time()
bench_out = model.generate(
    **bench_input,
    max_new_tokens=128,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)
torch.cuda.synchronize() if device == "cuda" else None
elapsed = time.time() - start

new_tokens = bench_out.shape[-1] - bench_input.input_ids.shape[-1]
print(f"Generated {new_tokens} tokens in {elapsed:.2f} s")
print(f"Throughput: {new_tokens / elapsed:.1f} tok/s")
print("\nSample output:")
print(tokenizer.decode(bench_out[0], skip_special_tokens=True))

### Swapping Models

Because a T4 still has ~16 GB of VRAM you can also try slightly larger models.
A few drop-in replacements (change `MODEL_ID` above):

| Model | Size | VRAM (bf16) | Notes |
|-------|------|-------------|-------|
| `Qwen/Qwen2.5-1.5B-Instruct` | 1.5 B | ~3 GB | Default in this notebook |
| `HuggingFaceTB/SmolLM2-1.7B-Instruct` | 1.7 B | ~3.5 GB | Very fast, compact |
| `google/gemma-2-2b-it` | 2 B | ~4.5 GB | Strong reasoning for size |
| `microsoft/Phi-4-mini-instruct` | 3.8 B | ~8 GB | Latest Phi, high quality |
| `Qwen/Qwen2.5-7B-Instruct` | 7 B | ~14 GB | Fits T4 tightly; use 4-bit below |

For models that do not fit in native 16-bit, load them in 4-bit quantisation:
```python
from transformers import BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
device_map="auto")
```

### Exercises

1. **Long context**: Increase the `max_new_tokens` to 1024 and observe whether the T4
     runs out of memory.
2. **Different model**: Swap in `google/gemma-2-2b-it` and compare answer quality on
     the same prompt.
3. **4-bit quantisation**: Try loading `Qwen2.5-7B-Instruct` in 4-bit and compare
     speed vs the 1.5 B native model.
4. **Streaming**: Wrap `model.generate` in a `TextIteratorStreamer` from
     `transformers` to print tokens as they arrive.


<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
